## Extracción de datos del IPC desde la API del INE

### Objetivo
Este notebook tiene como finalidad la extracción, exploración y estructuración de los datos del **Índice de Precios de Consumo (IPC)** publicados por el **Instituto Nacional de Estadística (INE)** a través de su API pública TEMPUS. 

A partir de la serie de datos IPCA (IPC Armonizado) con periodicidad mensual, se construye un **modelo dimensional** compuesto por:
- **Dimensiones**: tabla de tiempo (*tiempo*), tabla de territorios (*territorio*), tabla de sectores IPC (*sectores_ipc*) y tabla de tipos de medida (*tipo_medida*).
- **Tabla de hechos**: tabla central (*ipc*) que relaciona cada observación del índice con sus dimensiones asociadas.

### Metodología
1. **Conexión a la API del INE** — Llamada al endpoint `/DATOS_TABLA/76136` para obtener los últimos 60 periodos mensuales.
2. **Extracción de dimensiones** — Recorrido de la respuesta JSON para poblar las tablas dimensionales, garantizando unicidad mediante filtros de clave.
3. **Construcción de la tabla de hechos** — Cruce de las dimensiones a través de identificadores primarios y asignación del valor del IPC a cada combinación (territorio, sector, medida, periodo).
4. **Exportación** — Volcado de cada tabla a ficheros CSV en `../files/data_raw/` para su consumo en etapas posteriores del pipeline.

### Contexto del proyecto
Estos datos se integran en un análisis más amplio sobre la **resiliencia empresarial en España**, donde se combinarán con información de constitución y disolución de empresas para estudiar la correlación entre el entorno macroeconómico (inflación) y la actividad empresarial.

In [1]:
# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

import pandas as pd
# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación del módulo de conexión a la API
from src.api import connection_api as conexion_api
from src.api.config import API_URLS


In [2]:
url = API_URLS["ipc"]

In [3]:
data = conexion_api.llamada_api(url)

In [4]:
len(data)

1120

In [5]:
data[70]['Nombre']    #Ruta al nombre

'Andalucía. Vestido y calzado. Variación anual. '

In [6]:
data[0]['Data'][0]['Periodo']['Mes_inicio']     #Ruta al mes

'5'

In [7]:
data[0]['Data'][0]['Anyo']       #Ruta al año

2026

In [8]:
data[0]['Data'][0]['Periodo']['Nombre_largo']       #Ruta al nombre del mes

'Mayo'

In [9]:
data[0]['Data'][0]['Valor']             #Ruta al IPC

102.951

In [10]:
data[0]['Data']

[{'Fecha': 1777586400000,
  'TipoDato': {'Id': 1, 'Nombre': 'Definitivo', 'Codigo': 'D'},
  'Periodo': {'Id': 5,
   'Valor': 5,
   'FK_Periodicidad': 1,
   'Dia_inicio': '1',
   'Mes_inicio': '5',
   'Codigo': '05',
   'Nombre': 'M05',
   'Nombre_largo': 'Mayo'},
  'Anyo': 2026,
  'NombrePeriodo': '2026M05',
  'CodigoPeriodo': '202605',
  'Valor': 102.951,
  'Secreto': False},
 {'Fecha': 1774994400000,
  'TipoDato': {'Id': 1, 'Nombre': 'Definitivo', 'Codigo': 'D'},
  'Periodo': {'Id': 4,
   'Valor': 4,
   'FK_Periodicidad': 1,
   'Dia_inicio': '1',
   'Mes_inicio': '4',
   'Codigo': '04',
   'Nombre': 'M04',
   'Nombre_largo': 'Abril'},
  'Anyo': 2026,
  'NombrePeriodo': '2026M04',
  'CodigoPeriodo': '202604',
  'Valor': 102.883,
  'Secreto': False},
 {'Fecha': 1772319600000,
  'TipoDato': {'Id': 1, 'Nombre': 'Definitivo', 'Codigo': 'D'},
  'Periodo': {'Id': 3,
   'Valor': 3,
   'FK_Periodicidad': 1,
   'Dia_inicio': '1',
   'Mes_inicio': '3',
   'Codigo': '03',
   'Nombre': 'M03',
   

In [13]:
tiempo = {'id_tiempo': [], 'anio': [], 'mes': [], 'nombre_mes': [] }

for serie in data:
    for dato in serie['Data']:
        if dato['CodigoPeriodo'] not in tiempo['id_tiempo']:
            tiempo['id_tiempo'].append(dato['CodigoPeriodo'])
            tiempo['anio'].append(dato['Anyo'])
            tiempo['mes'].append(dato['Periodo']['Mes_inicio'])
            tiempo['nombre_mes'].append(dato['Periodo']['Nombre_largo'])


In [14]:
tiempo = pd.DataFrame(tiempo)
tiempo.sample()

,id_tiempo,anio,mes,nombre_mes
256,200501,2005,1,Enero


In [15]:
tiempo.shape

(294, 4)

In [16]:
tiempo.to_csv('../files/data_raw/tiempo.csv', index=False)

In [17]:
territorio = {'id_territorio': [], 'nombre_territorio': []}
id_ter = 1

for serie in data:
    nombre_completo = serie['Nombre']
    nombre_limpio = nombre_completo.split('.')[0].strip()

    if nombre_limpio not in territorio['nombre_territorio']:
        territorio['id_territorio'].append(id_ter)
        territorio['nombre_territorio'].append(nombre_limpio)

        id_ter += 1

In [18]:
territorio = pd.DataFrame(territorio)
territorio.sample()

,id_territorio,nombre_territorio
3,4,"Asturias, Principado de"


In [19]:
territorio.shape

(20, 2)

In [20]:
territorio.to_csv('../files/data_raw/territorio.csv', index=False)

In [21]:
sectores_ipc = {'id_sector': [], 'nombre_sector': []}
id_sec = 1

for serie in data:
    nombre_completo = serie['Nombre']
    sector_limpio = nombre_completo.split('.')[1].strip()

    if sector_limpio not in sectores_ipc['nombre_sector']:
        sectores_ipc['id_sector'].append(id_sec)
        sectores_ipc['nombre_sector'].append(sector_limpio)

        id_sec += 1        

In [22]:
sectores_ipc = pd.DataFrame(sectores_ipc)

In [23]:
sectores_ipc

,id_sector,nombre_sector
0,1,Índice general
1,2,Alimentos y bebidas no alcohólicas
2,3,Bebidas alcohólicas y tabaco
3,4,Vestido y calzado
4,5,"Vivienda, agua, electricidad, gas y otros comb..."
5,6,"Muebles, artículos del hogar y artículos para ..."
6,7,Sanidad
7,8,Transporte
8,9,Información y comunicaciones
9,10,"Actividades recreativas, deporte y cultura"


In [24]:
sectores_ipc.to_csv('../files/data_raw/sectores_ipc.csv', index=False)

In [25]:
tipo_medida = {'id_medida': [], 'nombre_medida': []}
id_med = 1

for serie in data:
    nombre_completo = serie['Nombre']
    medida_limpio = nombre_completo.split('.')[2].strip()

    if medida_limpio not in tipo_medida['nombre_medida']:
        tipo_medida['id_medida'].append(id_med)
        tipo_medida['nombre_medida'].append(medida_limpio)

        id_med += 1     

In [26]:
tipo_medida

{'id_medida': [1, 2, 3, 4],
 'nombre_medida': ['Índice',
  'Variación mensual',
  'Variación anual',
  'Variación en lo que va de año']}

In [27]:
tipo_medida = pd.DataFrame(tipo_medida)

In [28]:
tipo_medida

,id_medida,nombre_medida
0,1,Índice
1,2,Variación mensual
2,3,Variación anual
3,4,Variación en lo que va de año


In [29]:
tipo_medida.to_csv('../files/data_raw/tipo_medida.csv', index=False)

In [ ]:
#  Creamos diccionarios

map_territorios = dict(zip(territorio['nombre_territorio'], territorio['id_territorio']))
map_sectores = dict(zip(sectores_ipc['nombre_sector'], sectores_ipc['id_sector']))
map_medidas = dict(zip(tipo_medida['nombre_medida'], tipo_medida['id_medida']))
 
ipc = {
    'id_tiempo': [],
    'id_territorio': [],
    'id_sector': [],
    'id_medida': [],
    'valor_ipc': []
}
 
# Extracción y Cruce de IDs
for serie in data:
    nombre_completo = serie['Nombre']
    partes = nombre_completo.split('.')
    if len(partes) >= 3:

        # Extraemos los nombres de texto limpio
        territorio_texto = partes[0].strip()
        sector_texto = partes[1].strip()
        medida_texto = partes[2].strip()

        # Miramos en nuestros mapas qué ID le toca a cada texto
        id_terr = map_territorios.get(territorio_texto)
        id_sec = map_sectores.get(sector_texto)
        id_med = map_medidas.get(medida_texto)

        # Si por algún motivo el INE da un texto que no guardamos antes, nos lo saltamos
        if id_terr is None or id_sec is None or id_med is None:
            continue

        # Bajamos al segundo nivel: los datos históricos de esta serie
        for dato in serie['Data']:
            valor = dato.get('Valor')
            id_time = dato.get('CodigoPeriodo')     # Sacamos el código del periodo para usarlo como ID de tiempo

            # Si tenemos todos los datos, appendeamos en la tabla
            if valor is not None and id_time is not None:
                ipc['id_tiempo'].append(str(id_time))
                ipc['id_territorio'].append(id_terr)
                ipc['id_sector'].append(id_sec)
                ipc['id_medida'].append(id_med)
                ipc['valor_ipc'].append(valor)
 
 

In [31]:
print(map_territorios)

{'Nacional': 1, 'Andalucía': 2, 'Aragón': 3, 'Asturias, Principado de': 4, 'Balears, Illes': 5, 'Canarias': 6, 'Cantabria': 7, 'Castilla y León': 8, 'Castilla - La Mancha': 9, 'Cataluña': 10, 'Comunitat Valenciana': 11, 'Extremadura': 12, 'Galicia': 13, 'Madrid, Comunidad de': 14, 'Murcia, Región de': 15, 'Navarra, Comunidad Foral de': 16, 'País Vasco': 17, 'Rioja, La': 18, 'Ceuta': 19, 'Melilla': 20}


In [32]:
print(ipc) # Print de control

{'id_tiempo': ['202605', '202604', '202603', '202602', '202601', '202512', '202511', '202510', '202509', '202508', '202507', '202506', '202505', '202504', '202503', '202502', '202501', '202412', '202411', '202410', '202409', '202408', '202407', '202406', '202405', '202404', '202403', '202402', '202401', '202312', '202311', '202310', '202309', '202308', '202307', '202306', '202305', '202304', '202303', '202302', '202301', '202212', '202211', '202210', '202209', '202208', '202207', '202206', '202205', '202204', '202203', '202202', '202201', '202112', '202111', '202110', '202109', '202108', '202107', '202106', '202105', '202104', '202103', '202102', '202101', '202012', '202011', '202010', '202009', '202008', '202007', '202006', '202005', '202004', '202003', '202002', '202001', '201912', '201911', '201910', '201909', '201908', '201907', '201906', '201905', '201904', '201903', '201902', '201901', '201812', '201811', '201810', '201809', '201808', '201807', '201806', '201805', '201804', '2018

In [33]:
ipc = pd.DataFrame(ipc)

ipc.sample(5)

,id_tiempo,id_territorio,id_sector,id_medida,valor_ipc
138298,202605,9,7,1,101.694
83445,200702,6,2,1,62.437
91944,200612,6,9,2,-1.800
174486,201312,11,9,4,-6.900
327772,201001,20,14,3,2.800


In [34]:
ipc.to_csv('../files/data_raw/ipc.csv', index=False)